# Experiment 04: Error Analysis & Confidence Calibration

This notebook performs rigorous post-hoc error analysis on the best-performing model.

Sections:
1. Load model and test predictions
2. Most confused misconception pairs
3. Classes with lowest recall
4. Classes with highest false positives
5. High-confidence wrong predictions
6. Low-confidence (uncertain) predictions
7. Systematic code-pattern confusion analysis
8. Confidence calibration analysis
9. Temperature scaling calibration

In [ ]:
import os, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.calibration import calibration_curve

sys.path.insert(0, '..')

# Paths
DATA_DIR    = '../data/processed'
MODELS_DIR  = '../models'
RESULTS_DIR = '../experiments/results'

# Load all experiment results
results = {}
for fn in os.listdir(RESULTS_DIR):
    if fn.endswith('.json'):
        with open(os.path.join(RESULTS_DIR, fn)) as f:
            results[fn.replace('.json', '')] = json.load(f)

print('Available experiments:', list(results.keys()))

## 1. Summary of All Experiments

In [ ]:
summary_rows = []
for name, r in sorted(results.items()):
    summary_rows.append({
        'Experiment': name,
        'Macro F1': r.get('macro_f1', 0),
        'Weighted F1': r.get('weighted_f1', 0),
        'Top-1 Acc': r.get('accuracy', 0),
        'Top-3 Acc': r.get('top3_accuracy', 0),
        'Macro Prec': r.get('macro_precision', 0),
        'Macro Rec': r.get('macro_recall', 0),
        'N Samples': r.get('n_samples', 0),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Macro F1', ascending=False)
print(summary_df.to_string(index=False))

## 2. Load Best Model and Generate Predictions

In [ ]:
# Determine best model on validation Macro F1
val_results = {k: v for k, v in results.items() if '_val' in k}
if val_results:
    best_name = max(val_results, key=lambda k: val_results[k].get('macro_f1', 0))
    print(f'Best experiment on validation: {best_name}')
    print(f'  Macro F1: {val_results[best_name]["macro_f1"]:.4f}')
else:
    print('No validation results found — run experiments first.')
    best_name = None

In [ ]:
# Load the best model
# Priority: exp5_fusion > exp2_minilm > exp1_tfidf
model_priority = [
    ('exp5_fusion.pkl',         'exp5_fusion'),
    ('exp2_minilm_svm.pkl',     'exp2_minilm'),
    ('exp1_tfidf_logreg.pkl',   'exp1_tfidf'),
]

best_model_file = None
for fname, tag in model_priority:
    fp = os.path.join(MODELS_DIR, fname)
    if os.path.exists(fp):
        best_model_file = fp
        best_tag = tag
        break

if best_model_file is None:
    print('No model found — run training scripts first.')
else:
    artefact = joblib.load(best_model_file)
    clf = artefact['clf']
    le  = artefact['le']
    tfidf = artefact.get('tfidf', None)
    print(f'Loaded model: {best_tag} from {best_model_file}')
    print(f'Number of classes: {len(le.classes_)}')

In [ ]:
# Generate test-set predictions
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
codes = test_df['generated_code'].fillna('').tolist()
y_true_raw = test_df['global_misconception_index'].values

# Only evaluate on classes seen during training
mask = np.isin(y_true_raw, le.classes_)
codes_f = [c for c, m in zip(codes, mask) if m]
y_true  = le.transform(y_true_raw[mask])

if tfidf is not None:
    X = tfidf.transform(codes_f)
else:
    # Try to load cached embeddings for this split
    emb_cache = os.path.join(MODELS_DIR, f'{best_tag}_svm_emb_test.npy')
    if os.path.exists(emb_cache):
        from sklearn.preprocessing import normalize
        X_emb = normalize(np.load(emb_cache)[mask])
        # Try to get tree-sitter features
        ts_cache = os.path.join(MODELS_DIR, 'ts_feats_test.npy')
        if os.path.exists(ts_cache):
            X_ts = np.load(ts_cache)[mask]
            X = np.hstack([X_emb, X_ts])
        else:
            X = X_emb
    else:
        print('No cached embedding found — falling back to TF-IDF model')
        tfidf_art = joblib.load(os.path.join(MODELS_DIR, 'exp1_tfidf_logreg.pkl'))
        clf = tfidf_art['clf']; le = tfidf_art['le']; tfidf_v = tfidf_art['tfidf']
        X = tfidf_v.transform(codes_f)

y_pred  = clf.predict(X)
y_proba = clf.predict_proba(X)
confidence = np.max(y_proba, axis=1)

print(f'Test samples evaluated: {len(y_true)}')
print(f'Accuracy: {(y_true == y_pred).mean():.4f}')

## 3. Most Confused Misconception Pairs

In [ ]:
classes_in_test = sorted(set(y_true))
cm = confusion_matrix(y_true, y_pred, labels=classes_in_test)

# Find the most confused off-diagonal pairs
cm_df = pd.DataFrame(cm, index=le.inverse_transform(classes_in_test),
                     columns=le.inverse_transform(classes_in_test))

confused_pairs = []
for i, true_cls in enumerate(classes_in_test):
    for j, pred_cls in enumerate(classes_in_test):
        if i != j and cm[i, j] > 0:
            confused_pairs.append({
                'true_misc': int(le.inverse_transform([true_cls])[0]),
                'pred_misc': int(le.inverse_transform([pred_cls])[0]),
                'count': int(cm[i, j])
            })

confused_df = pd.DataFrame(confused_pairs).sort_values('count', ascending=False)
print('Top 15 most confused misconception pairs:')
print(confused_df.head(15).to_string(index=False))

## 4. Per-Class Recall Analysis (Classes with Lowest Recall)

In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score

class_report = classification_report(
    y_true, y_pred, labels=classes_in_test, output_dict=True, zero_division=0
)

per_class_rows = []
for cls_enc in classes_in_test:
    cls_raw = int(le.inverse_transform([cls_enc])[0])
    rep = class_report.get(str(cls_enc), {})
    n_true = int(np.sum(y_true == cls_enc))
    per_class_rows.append({
        'misconception_id': cls_raw,
        'n_true': n_true,
        'precision': rep.get('precision', 0),
        'recall':    rep.get('recall', 0),
        'f1':        rep.get('f1-score', 0),
    })

per_class_df = pd.DataFrame(per_class_rows)
print('Classes with lowest recall (bottom 10):')
print(per_class_df.sort_values('recall').head(10).to_string(index=False))

# Visualise
fig, ax = plt.subplots(figsize=(14, 5))
sorted_df = per_class_df.sort_values('misconception_id')
ax.bar(sorted_df['misconception_id'].astype(str), sorted_df['recall'], color='steelblue', alpha=0.8)
ax.set_title('Per-Class Recall on Test Set')
ax.set_xlabel('Misconception ID')
ax.set_ylabel('Recall')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig('../experiments/results/per_class_recall.png', dpi=100)
plt.show()

## 5. High-Confidence Wrong Predictions

In [ ]:
wrong_mask = y_pred != y_true
high_conf_wrong = np.argsort(confidence[wrong_mask])[::-1][:5]
wrong_indices = np.where(wrong_mask)[0][high_conf_wrong]

print('Top 5 high-confidence WRONG predictions:')
for idx in wrong_indices:
    true_class = int(le.inverse_transform([y_true[idx]])[0])
    pred_class = int(le.inverse_transform([y_pred[idx]])[0])
    conf = confidence[idx]
    code_snippet = codes_f[idx][:200].replace('\n', ' | ')
    print(f'\n  True: {true_class} | Pred: {pred_class} | Conf: {conf:.3f}')
    print(f'  Code: {code_snippet}...')

## 6. Low-Confidence (Uncertain) Predictions

In [ ]:
low_conf_idx = np.argsort(confidence)[:5]
print('Top 5 LOWEST confidence predictions:')
for idx in low_conf_idx:
    true_class = int(le.inverse_transform([y_true[idx]])[0])
    pred_class = int(le.inverse_transform([y_pred[idx]])[0])
    conf = confidence[idx]
    correct = '(CORRECT)' if y_pred[idx] == y_true[idx] else '(WRONG)'
    print(f'  True: {true_class} | Pred: {pred_class} | Conf: {conf:.3f} {correct}')

## 7. Correctness vs Confidence Distribution

In [ ]:
correct_mask = y_pred == y_true
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(confidence[correct_mask],  bins=20, alpha=0.7, label='Correct',  color='green')
axes[0].hist(confidence[~correct_mask], bins=20, alpha=0.7, label='Incorrect', color='red')
axes[0].set_title('Confidence Distribution: Correct vs Incorrect')
axes[0].set_xlabel('Max Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].legend()

# Reliability diagram (calibration curve)
# Binarize: is top-1 prediction correct?
binary_correct = correct_mask.astype(int)
try:
    fraction_of_positives, mean_predicted_value = calibration_curve(
        binary_correct, confidence, n_bins=10, strategy='quantile'
    )
    axes[1].plot(mean_predicted_value, fraction_of_positives, 's-', label='Model', color='blue')
    axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    axes[1].set_title('Reliability Diagram (Calibration Curve)')
    axes[1].set_xlabel('Mean Predicted Probability')
    axes[1].set_ylabel('Fraction Correct')
    axes[1].legend()
except Exception as e:
    print('Calibration curve error:', e)

plt.tight_layout()
plt.savefig('../experiments/results/confidence_analysis.png', dpi=100)
plt.show()

## 8. Temperature Scaling Calibration

Temperature scaling divides the logits by a scalar T before softmax.
T > 1 makes predictions more uncertain (spreads distribution).
T < 1 makes predictions more confident (sharpens distribution).

We tune T on the validation set and report calibration improvement.

In [ ]:
# Load validation predictions for tuning temperature
val_df = pd.read_csv(os.path.join(DATA_DIR, 'validation.csv'))
val_codes = val_df['generated_code'].fillna('').tolist()
y_val_raw = val_df['global_misconception_index'].values

val_mask = np.isin(y_val_raw, le.classes_)
val_codes_f = [c for c, m in zip(val_codes, val_mask) if m]
y_val = le.transform(y_val_raw[val_mask])

if tfidf is not None:
    X_val = tfidf.transform(val_codes_f)
else:
    emb_cache_v = os.path.join(MODELS_DIR, f'{best_tag}_svm_emb_val.npy')
    if os.path.exists(emb_cache_v):
        from sklearn.preprocessing import normalize
        X_emb_v = normalize(np.load(emb_cache_v)[val_mask])
        ts_cache_v = os.path.join(MODELS_DIR, 'ts_feats_val.npy')
        if os.path.exists(ts_cache_v):
            X_ts_v = np.load(ts_cache_v)[val_mask]
            X_val = np.hstack([X_emb_v, X_ts_v])
        else:
            X_val = X_emb_v
    else:
        # Fall back to TF-IDF
        tfidf_art = joblib.load(os.path.join(MODELS_DIR, 'exp1_tfidf_logreg.pkl'))
        X_val = tfidf_art['tfidf'].transform(val_codes_f)

val_proba = clf.predict_proba(X_val)
print(f'Val samples: {len(y_val)}')

In [ ]:
from scipy.special import softmax
from scipy.optimize import minimize_scalar

def apply_temperature(proba: np.ndarray, T: float) -> np.ndarray:
    """Apply temperature scaling to probability matrix."""
    log_proba = np.log(np.clip(proba, 1e-12, 1.0))
    scaled = log_proba / T
    return softmax(scaled, axis=1)

def nll_loss(T, proba, y_true):
    """Negative log-likelihood under temperature T."""
    calibrated = apply_temperature(proba, T)
    return -np.mean(np.log(np.clip(calibrated[np.arange(len(y_true)), y_true], 1e-12, 1)))

# Tune T on validation
result = minimize_scalar(nll_loss, bounds=(0.1, 10.0), method='bounded',
                         args=(val_proba, y_val))
best_T = result.x
print(f'Optimal temperature T = {best_T:.4f}')

# Apply calibration to test predictions
y_proba_cal = apply_temperature(y_proba, best_T)
confidence_cal = np.max(y_proba_cal, axis=1)

# Compare calibration
binary_correct_test = (y_pred == y_true).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
try:
    fop_before, mpv_before = calibration_curve(binary_correct_test, confidence, n_bins=8, strategy='quantile')
    fop_after,  mpv_after  = calibration_curve(binary_correct_test, confidence_cal, n_bins=8, strategy='quantile')
    axes[0].plot(mpv_before, fop_before, 's-', label='Before T-scaling', color='red')
    axes[0].plot(mpv_after,  fop_after,  's-', label=f'After T={best_T:.2f}', color='green')
    axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect')
    axes[0].set_title('Calibration: Before vs After Temperature Scaling')
    axes[0].set_xlabel('Mean Predicted Probability')
    axes[0].set_ylabel('Fraction Correct')
    axes[0].legend()
except Exception as e:
    print('Calibration curve error:', e)

axes[1].hist(confidence,     bins=20, alpha=0.6, label='Before', color='red')
axes[1].hist(confidence_cal, bins=20, alpha=0.6, label='After',  color='green')
axes[1].set_title('Confidence Distribution Before/After Temperature Scaling')
axes[1].set_xlabel('Max Predicted Probability')
axes[1].set_ylabel('Count')
axes[1].legend()
plt.tight_layout()
plt.savefig('../experiments/results/calibration_comparison.png', dpi=100)
plt.show()

# Save temperature
with open('../experiments/results/temperature_scaling.json', 'w') as f:
    json.dump({'optimal_temperature': best_T, 'tuned_on': 'validation_set'}, f, indent=2)
print('Temperature saved.')

## 9. Model Selection Decision

In [ ]:
print('='*70)
print('MODEL SELECTION SUMMARY')
print('='*70)
if val_results:
    for name in sorted(val_results, key=lambda k: val_results[k].get('macro_f1', 0), reverse=True):
        r = val_results[name]
        print(f'{name:<40}  Macro F1={r["macro_f1"]:.4f}  Top3={r["top3_accuracy"]:.4f}')
print()
print('Decision criteria:')
print('  1. Macro F1 (primary metric — accounts for imbalance)')
print('  2. Top-3 accuracy (useful for Re:Learn to show alternatives)')
print('  3. Problem-disjoint generalisation (robustness check)')
print('  4. Confidence calibration (required for uncertainty estimate)')
print('  5. Inference speed (must support real-time FastAPI)')